# Introduction to the OpenAI Responses API

The **Responses API** is OpenAI's primary API surface. It combines the strengths of Chat Completions and the old Assistants API, and it is where the modern features live: durable **conversations**, server-side **compaction**, response **phases**, hosted tools, connectors, and WebSocket streaming.

This notebook rebuilds the introduction around the **Conversations API** as the primary multi-turn pattern, and reframes Chat Completions as legacy.

> **Model update (GPT-5.6 refresh).** All `model=` calls target **`gpt-5.6-sol`** (the bare `gpt-5.6` alias also routes to Sol), the current flagship model for coding and professional work. `reasoning.effort` is pinned explicitly where cost/latency matters (GPT-5.6 defaults to `medium`). Lighter cells may use `gpt-5.6-luna`, the cheapest, lowest-latency tier — the API name is confirmed available.

## Setup

In [1]:
import os
import getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")

In [2]:
%pip install --upgrade openai pandas jinja2 pydantic

  Using cached pandas-3.0.5-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)


Using cached pandas-3.0.5-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (11.3 MB)


  Attempting uninstall: pandas
    Found existing installation: pandas 2.3.2


    Uninstalling pandas-2.3.2:
      Successfully uninstalled pandas-2.3.2


Note: you may need to restart the kernel to use updated packages.


## Initialize the Client

In [3]:
from openai import OpenAI

client = OpenAI()
# Make sure your OPENAI_API_KEY environment variable is set

## Basic Text Response

A single, stateless call. `instructions` sets behavior; `input` is the user turn.

In [4]:
response = client.responses.create(
    model="gpt-5.6-sol",
    instructions="You are a coding assistant that talks like a pirate.",
    input="How do I check if a Python object is an instance of a class?",
    reasoning={"effort": "low"},
)

print(response.output_text)

Arrr, use Python’s built-in `isinstance()`:

```python
if isinstance(obj, MyClass):
    print("obj is an instance of MyClass")
```

You can check against multiple classes using a tuple:

```python
if isinstance(obj, (ClassA, ClassB)):
    print("obj is an instance of ClassA or ClassB")
```

`isinstance()` also recognizes subclasses. To check only the exact class, use:

```python
if type(obj) is MyClass:
    print("obj is exactly a MyClass")
```

Prefer `isinstance()` in most cases, matey.


## Multi-Turn with the Conversations API (the primary pattern)

Instead of manually re-sending history each turn, create a **durable conversation object** with `client.conversations.create()`. It stores messages, tool calls, and tool outputs under its own id. Pass that id on each `responses.create()` call and the model shares context automatically.

Conversations persist **indefinitely** (no 30-day TTL), unlike standalone `store: true` responses which expire after 30 days.

In [5]:
# 1) Create a durable conversation
conversation = client.conversations.create()
print("conversation id:", conversation.id)

# 2) First turn
r1 = client.responses.create(
    model="gpt-5.6-sol",
    conversation=conversation.id,
    input="I'm designing a URL shortener. What storage would you start with?",
    reasoning={"effort": "medium"},
)
print("Turn 1:", r1.output_text[:200], "...")

# 3) Second turn -- no manual history; the conversation id carries context
r2 = client.responses.create(
    model="gpt-5.6-sol",
    conversation=conversation.id,
    input="Now how would I add custom vanity slugs to that design?",
    reasoning={"effort": "medium"},
)
print("\nTurn 2:", r2.output_text[:200], "...")

conversation id: conv_6a66728ab730819693cb3c5c9493646f0a5fc6b967479120


Turn 1: I’d start with **PostgreSQL as the source of truth**, optionally adding **Redis as a read-through cache**.

### Why PostgreSQL
A URL shortener has a simple access pattern—lookup by short code—but also ...



Turn 2: Treat vanity slugs as user-selected values in the same namespace as generated short codes. The existing primary-key uniqueness constraint handles concurrency safely.

### Schema

If links can also use ...


### Legacy chaining: `previous_response_id`

Before conversations, you chained turns with `previous_response_id`. It still works for `gpt-5.6-sol`, but prefer conversations for durable, multi-turn state.

```python
follow_up = client.responses.create(
    model="gpt-5.6-sol",
    input="...next turn...",
    previous_response_id=r1.id,   # legacy chaining
)
```

Billing note (both patterns): prior input tokens in the chain are re-billed as input on each turn.

## Server-Side Compaction (`/responses/compact`)

Long conversations grow expensive because prior tokens are re-billed each turn. The Responses API can **compact** context server-side — shrinking what you send per turn while preserving meaning.

- **Standalone:** `client.responses.compact(model=..., input=[...])` — takes a full context window and returns a compacted version to use as input on the next call.
- **Inline:** pass `context_management=[{"type": "compaction", "compact_threshold": N}]` to `responses.create()` — the server compacts automatically when the token count crosses `N`.

In [6]:
# Inline compaction: the server compacts context automatically when the token count crosses the threshold.
# context_management takes a list with a "compaction" entry specifying the token threshold.
r3 = client.responses.create(
    model="gpt-5.6-sol",
    conversation=conversation.id,
    input="Summarize the whole design so far in 5 bullets.",
    reasoning={"effort": "low"},
    context_management=[{"type": "compaction", "compact_threshold": 200_000}],
)
print(r3.output_text)

- Use **PostgreSQL as the source of truth**, with **Redis as a read-through cache** for fast redirects.
- Store each mapping as `(domain_id, slug) → target_url`, with creation/expiration metadata and optional owner information.
- Support both generated codes and vanity slugs in the same namespace; enforce uniqueness with a composite primary key and return **409 Conflict** on collisions.
- Normalize and validate vanity slugs, reserve application routes, rate-limit creation, and preferably keep slugs immutable or quarantine deleted ones.
- Resolve redirects through Redis first, then PostgreSQL; invalidate cache entries on updates and send click analytics asynchronously outside the redirect path.


## Streaming Responses

Stream a response and process `response.output_text.delta` events to accumulate text incrementally as the model generates it.

In [7]:
# Stream a response and print output text incrementally as it arrives.
stream = client.responses.create(
    model="gpt-5.6-sol",
    input="Plan and then write a Python function to validate an email address.",
    reasoning={"effort": "medium"},
    stream=True,
)

for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
print()

##

 Plan

1

.

 Confirm

...

 that

 a

 mailbox

 exists

.

## Transport: WebSocket Mode (low-latency streaming)

For latency-sensitive apps, the Responses API supports a **WebSocket transport**. It keeps a connection-local cache so follow-up turns (via `previous_response_id`) continue with very low latency -- ideal for voice and live agent UIs.

We don't open a socket here (it needs a running event loop and the realtime/ws client), but the pattern is: open a WS connection, send response requests over it, and read streamed events back on the same connection. See the [OpenAI Realtime API docs](https://platform.openai.com/docs/guides/realtime) for a concrete WebSocket connection pattern.

## Image Analysis

In [8]:
from IPython.display import display, Image

img_url = "https://commons.wikimedia.org/w/index.php?title=Special:FilePath&file=2023_06_08_Raccoon1.jpg&width=800"

# Visualize the image from the URL in this notebook
display(Image(url=img_url))

In [9]:
import requests
import base64



resp = requests.get(img_url, headers={"User-Agent": "Mozilla/5.0"}, allow_redirects=True)

prompt = "Extract all the visual elements of this image into a bullet points list, just output that list."

resp.raise_for_status()
img_data_url = f"data:image/jpeg;base64,{base64.b64encode(resp.content).decode()}"

response = client.responses.create(
    model="gpt-5.6-sol",
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": prompt},
                {"type": "input_image", "image_url": img_data_url},
            ],
        }
    ],
    reasoning={"effort": "low"},
)

print(response.output_text)

- Large, rough-textured tree trunk dominating the left and lower portions of the image
- Broken or hollowed tree section extending across the lower right
- Raccoon peeking out from behind the tree
- Raccoon’s pointed ears with light-colored edges
- Distinctive black facial mask around the eyes
- White and gray facial markings
- Black nose and visible whiskers
- Gray-brown fur
- Dark, shadowy forest background
- Faint patches of green foliage in the background
- Sunlight illuminating the raccoon and tree bark
- Strong contrast between the lit foreground and dark background


## Streaming Responses

In [10]:
stream = client.responses.create(
    model="gpt-5.6-sol",
    input="Write a one-sentence bedtime story about having pancakes as an afternoon snack.",
    reasoning={"effort": "none"},
    stream=True,
)

for event in stream:
    print(event)

ResponseCreatedEvent(response=Response(id='resp_098fefcab73b347c006a6672bc5518819e8b38d9bce45dff81', created_at=1785098940.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.6-sol', object='response', output=[], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=0.98, background=False, completed_at=None, conversation=None, max_output_tokens=None, max_tool_calls=None, moderation=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_options=None, prompt_cache_retention='24h', reasoning=Reasoning(context='all_turns', effort='none', generate_summary=None, mode='standard', summary=None), safety_identifier=None, service_tier='auto', status='in_progress', text=ResponseTextConfig(format=ResponseFormatText(type='text'), verbosity='medium'), top_logprobs=0, truncation='disabled', usage=None, user=None, frequency_penalty=0.0, presence_penalty=0.0, store=True, tool_usage={'image_gen': {'input_tokens': 0, 'inp

ResponseOutputItemAddedEvent(item=ResponseOutputMessage(id='msg_098fefcab73b347c006a6672bceb8c819e9e4a4a9389ea1baf', content=[], role='assistant', status='in_progress', type='message', phase='final_answer'), output_index=0, sequence_number=2, type='response.output_item.added')
ResponseContentPartAddedEvent(content_index=0, item_id='msg_098fefcab73b347c006a6672bceb8c819e9e4a4a9389ea1baf', output_index=0, part=ResponseOutputText(annotations=[], text='', type='output_text', logprobs=[]), sequence_number=3, type='response.content_part.added')
ResponseTextDeltaEvent(content_index=0, delta='One', item_id='msg_098fefcab73b347c006a6672bceb8c819e9e4a4a9389ea1baf', logprobs=[], output_index=0, sequence_number=4, type='response.output_text.delta', obfuscation='jDiQ8aT0S1gM4')
ResponseTextDeltaEvent(content_index=0, delta=' sleepy', item_id='msg_098fefcab73b347c006a6672bceb8c819e9e4a4a9389ea1baf', logprobs=[], output_index=0, sequence_number=5, type='response.output_text.delta', obfuscation='5MwBj

ResponseTextDeltaEvent(content_index=0, delta=' shared', item_id='msg_098fefcab73b347c006a6672bceb8c819e9e4a4a9389ea1baf', logprobs=[], output_index=0, sequence_number=9, type='response.output_text.delta', obfuscation='57XTXnymS')
ResponseTextDeltaEvent(content_index=0, delta=' a', item_id='msg_098fefcab73b347c006a6672bceb8c819e9e4a4a9389ea1baf', logprobs=[], output_index=0, sequence_number=10, type='response.output_text.delta', obfuscation='mFBQP6Lqs5SPc2')
ResponseTextDeltaEvent(content_index=0, delta=' stack', item_id='msg_098fefcab73b347c006a6672bceb8c819e9e4a4a9389ea1baf', logprobs=[], output_index=0, sequence_number=11, type='response.output_text.delta', obfuscation='CwXOiyulM3')
ResponseTextDeltaEvent(content_index=0, delta=' of', item_id='msg_098fefcab73b347c006a6672bceb8c819e9e4a4a9389ea1baf', logprobs=[], output_index=0, sequence_number=12, type='response.output_text.delta', obfuscation='2C77XlTmaNwbN')
ResponseTextDeltaEvent(content_index=0, delta=' moon', item_id='msg_098fe

ResponseTextDeltaEvent(content_index=0, delta=' pancakes', item_id='msg_098fefcab73b347c006a6672bceb8c819e9e4a4a9389ea1baf', logprobs=[], output_index=0, sequence_number=15, type='response.output_text.delta', obfuscation='XXo3Frh')
ResponseTextDeltaEvent(content_index=0, delta=' with', item_id='msg_098fefcab73b347c006a6672bceb8c819e9e4a4a9389ea1baf', logprobs=[], output_index=0, sequence_number=16, type='response.output_text.delta', obfuscation='eTPgNt7kh8c')
ResponseTextDeltaEvent(content_index=0, delta=' her', item_id='msg_098fefcab73b347c006a6672bceb8c819e9e4a4a9389ea1baf', logprobs=[], output_index=0, sequence_number=17, type='response.output_text.delta', obfuscation='aVgM1UEGFkqB')
ResponseTextDeltaEvent(content_index=0, delta=' teddy', item_id='msg_098fefcab73b347c006a6672bceb8c819e9e4a4a9389ea1baf', logprobs=[], output_index=0, sequence_number=18, type='response.output_text.delta', obfuscation='6eY4qYgroz')
ResponseTextDeltaEvent(content_index=0, delta=',', item_id='msg_098fefca

ResponseTextDeltaEvent(content_index=0, delta=' then', item_id='msg_098fefcab73b347c006a6672bceb8c819e9e4a4a9389ea1baf', logprobs=[], output_index=0, sequence_number=20, type='response.output_text.delta', obfuscation='e3jdMY5Y3yN')
ResponseTextDeltaEvent(content_index=0, delta=' drift', item_id='msg_098fefcab73b347c006a6672bceb8c819e9e4a4a9389ea1baf', logprobs=[], output_index=0, sequence_number=21, type='response.output_text.delta', obfuscation='lyJjKQV0Oo')
ResponseTextDeltaEvent(content_index=0, delta='ed', item_id='msg_098fefcab73b347c006a6672bceb8c819e9e4a4a9389ea1baf', logprobs=[], output_index=0, sequence_number=22, type='response.output_text.delta', obfuscation='1qFglNO6CIi5ET')
ResponseTextDeltaEvent(content_index=0, delta=' to', item_id='msg_098fefcab73b347c006a6672bceb8c819e9e4a4a9389ea1baf', logprobs=[], output_index=0, sequence_number=23, type='response.output_text.delta', obfuscation='5rwfO99xUwE8y')
ResponseTextDeltaEvent(content_index=0, delta=' bed', item_id='msg_098fe

ResponseTextDoneEvent(content_index=0, item_id='msg_098fefcab73b347c006a6672bceb8c819e9e4a4a9389ea1baf', logprobs=[], output_index=0, sequence_number=31, text='One sleepy afternoon, Mia shared a stack of moon-shaped pancakes with her teddy, then drifted to bed dreaming of syrupy stars.', type='response.output_text.done')
ResponseContentPartDoneEvent(content_index=0, item_id='msg_098fefcab73b347c006a6672bceb8c819e9e4a4a9389ea1baf', output_index=0, part=ResponseOutputText(annotations=[], text='One sleepy afternoon, Mia shared a stack of moon-shaped pancakes with her teddy, then drifted to bed dreaming of syrupy stars.', type='output_text', logprobs=[]), sequence_number=32, type='response.content_part.done')
ResponseOutputItemDoneEvent(item=ResponseOutputMessage(id='msg_098fefcab73b347c006a6672bceb8c819e9e4a4a9389ea1baf', content=[ResponseOutputText(annotations=[], text='One sleepy afternoon, Mia shared a stack of moon-shaped pancakes with her teddy, then drifted to bed dreaming of syrupy

## Structured Output with Pydantic

In [11]:
from openai import OpenAI

def process_image(prompt, img_url, model="gpt-5.6-sol"):
    response = client.responses.create(
        model=model,
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": prompt},
                    {"type": "input_image", "image_url": f"{img_url}"},
                ],
            }
        ],
        reasoning={"effort": "low"},
    )
    return response.output_text

In [12]:
from pydantic import BaseModel, Field

class ImageElements(BaseModel):
    subject: str = Field(description="The main subject of the image")
    general_background_description: str = Field(description="A general description of the background of the image")


def extract_structured_text(output):
    responses = client.responses.parse(
        model="gpt-5.6-sol",
        input=[
            {"role": "system", "content": "You extract the main subject and background contained in the input."},
            {"role": "user", "content": output},
        ],
        text_format=ImageElements,
    )
    return responses.output_parsed


photo_description = "A golden-brown pancake sits on a pristine white plate, its surface glistening with syrup and a pat of melting butter. The simple background highlights the pancake, making it the clear focus of the image."
extract_structured_text(photo_description)

ImageElements(subject='A golden-brown pancake topped with glistening syrup and a pat of melting butter on a pristine white plate.', general_background_description='A simple, uncluttered background that emphasizes the pancake as the clear focal point.')

In [13]:
from IPython.display import Markdown

food_url = "https://commons.wikimedia.org/w/index.php?title=Special:FilePath&file=Pizza_01.jpg&width=800"

display(Image(url=food_url))

In [14]:
from IPython.display import Markdown

food_url = "https://commons.wikimedia.org/w/index.php?title=Special:FilePath&file=Pizza_01.jpg&width=800"
resp = requests.get(food_url, headers={"User-Agent": "Mozilla/5.0"}, allow_redirects=True)
resp.raise_for_status()

image_of_food = f"data:image/jpeg;base64,{base64.b64encode(resp.content).decode()}"

prompt = "Extract the subject and the background of this image and output that into a bullet list."

output_processed_image = process_image(prompt, image_of_food)
structured_output = extract_structured_text(output_processed_image)

Markdown(f"""
**Here's a structured summary of the image:**
- **Subject:** {structured_output.subject}
- **Background:** {structured_output.general_background_description}
""")


**Here's a structured summary of the image:**
- **Subject:** A rustic pizza or flatbread topped with assorted roasted vegetables, including eggplant, artichokes, peppers, mushrooms, and leafy greens.
- **Background:** A white plate resting on a pale green-and-white patterned tablecloth.


## Legacy: the Chat Completions API

Chat Completions still works with `gpt-5.6-sol`, but it is now the **legacy** surface. The conversation-state, server-side compaction, response `phase`, connectors, hosted tools, and WebSocket features above are **Responses-API-first** and are not available through Chat Completions.

Use Chat Completions only when porting old code; new work should target the Responses API.

In [15]:
# Legacy Chat Completions (shown for migration reference only)
completion = client.chat.completions.create(
    model="gpt-5.6-sol",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Hello!"},
    ],
)
print(completion.choices[0].message.content)

Hello! How can I help you today?
